This part of the pipeline processes the raw Pseudofinder output and statistically compares the normalised pseudogene counts by order.

### Paths and parameters

#### Libraries

In [ ]:
import os
from os.path import join
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools as it
import scipy.stats as sts
import numpy as np
from statannotations.Annotator import Annotator

#### Pipeline input folders

In [ ]:
metadata_file = "dereplicated_metadata"
classification_file = "02-GTDB/filtered_classification_table"

#### Pipeline output folders

In [ ]:
task_root = "09-MGEs/pseudogenes"
output_folder = join(task_root, "output")
results_folder = join(task_root, "processed_output")

#### Tool pointers and parameters

#### Other setups

In [ ]:
custom_palette = sns.husl_palette()
custom_palette = [custom_palette[0], custom_palette[1], custom_palette[2], custom_palette[4]]
custom_palette

In [ ]:
os.makedirs(results_folder, exist_ok=True)

## Reading input files

### Parsing pseudofinder results

In [ ]:
result_dirs = os.listdir(output_folder)
hits = []
# Pseudogene hits are listed in the "*_pseudos.fasta" file, so counting the records ('>') in those files
for dir in result_dirs:
    dir_conts = os.listdir(join(output_folder, dir))
    pseudo_fastas = [f for f in dir_conts if '_pseudos.fasta' in f]
    for pfa in pseudo_fastas:
        with open(join(output_folder,  dir, pfa), "r") as handle:
            cont = handle.read()
            counts = cont.count('>')
        record = {'assembly_ID': dir, 'counts': counts}
        hits.append(record)
hits = pd.DataFrame(hits).set_index('assembly_ID')
hits

### GTDB orders

In [ ]:
classification = pd.read_table(classification_file, sep = "\t", header = None, names = ['accession', 'group'])
classification = classification.set_index('accession').squeeze()
classification

In [ ]:
plot_order = sorted(list(classification.unique()))

### Reading genome sizes

Necessary for normalising the pseudogene counts

In [ ]:
sizes = pd.read_table(metadata_file, usecols = [0,4], sep = "\t")
sizes = sizes.rename(columns = {'Assembly Stats Total Sequence Length': 'size',
                               'Assembly Accession': 'accession'})
sizes = sizes.set_index('accession').squeeze()
sizes

### Adding metadata columns

In [ ]:
hits = pd.merge(hits, classification, left_index = True, right_index = True)
hits = pd.merge(hits, sizes, left_index = True, right_index = True)
hits

In [ ]:
hits.to_csv(join(results_folder, "counts"), sep = '\t', index = False)

### General count plots

Normalise by genome size

In [ ]:
cluster_counts = hits.rename(columns = {'counts': 'No. pseudogenes'})
cluster_counts['Norm. no. pseudogenes'] = cluster_counts['No. pseudogenes']/cluster_counts['size']*1000000
cluster_counts

In [ ]:
cluster_counts.to_csv(join(results_folder, 'counts_per_assembly_cluster'), sep = '\t', index = False)

#### Barplot

In [ ]:
fig, ax = plt.subplots(figsize = (5,2))
ax = sns.barplot(ax = ax, data = cluster_counts, estimator = "mean", errorbar = "se",
                 x = "Norm. no. pseudogenes", y = "group", palette = custom_palette,
                 width = 0.9, orient = "h", order = plot_order)
plt.xlabel('Avg. norm. no. pseudogenes')
plt.ylabel('GTDB order')
plt.title('Pseudogenes')
plt.savefig(join(results_folder, "av_counts_pseudogenes_cluster_bar.svg"))
plt.show()

#### Violinplot

In [ ]:
fig, ax = plt.subplots(figsize = (5,3))
ax = sns.violinplot(ax = ax, data = cluster_counts, x = 'Norm. no. pseudogenes', y = 'group', 
                    palette = custom_palette, orient = 'h', cut = 0, order = plot_order)
plt.xlabel('Norm. no. pseudogenes')
plt.ylabel('GTDB order')
plt.title('Pseudogenes')

# Add statistical significance markers
pairs = list(it.combinations(cluster_counts['group'].unique(), 2))
annotator = Annotator(ax = ax, pairs = pairs, data = cluster_counts, x = 'Norm. no. pseudogenes', y = 'group', orient = 'h', cut = 0, 
                      order = plot_order)
annotator.configure(test = 'Brunner-Munzel', comparisons_correction="Bonferroni", text_format = 'star', loc = 'inside')
annotator.apply_and_annotate()

plt.savefig(join(results_folder, 'counts_pseudogenes_cluster_violin.svg'))
plt.show()

#### Exact stats

Getting all the counts grouped by order

In [ ]:
cluster_counts_stats = cluster_counts[['Norm. no. pseudogenes', 'group']].to_dict(orient = 'list')
cluster_counts_stats = list(zip(*cluster_counts_stats.values()))
counts_stats = {}
for record in cluster_counts_stats:
    try:
        counts_stats[record[1]].append(record[0])
    except KeyError:
        counts_stats[record[1]] = [record[0]]
counts_stats

In [ ]:
[(i, [np.mean(j), np.std(j)]) for i,j in counts_stats.items()]

In [ ]:
tests = list(it.combinations(counts_stats.keys(), 2)) # get all order pairs
for comb in tests:
    p_val = sts.brunnermunzel(counts_stats[comb[0]], counts_stats[comb[1]])[1]
    q_val = min(p_val * len(tests), 1) # Bonferroni correction
    print(f'{comb}: {q_val}')